# Sources and Generation

An `EnergySource` is the grid at the edge of the model. Generators are rotating machines, synchronous or
asynchronous, or inverter-based resources: a `PowerElectronicsConnection` with PV, battery or wind units
behind it.

CIM uses the load sign convention at a Terminal: positive `p` flows into the equipment. A generator that is
generating has negative `p` and, when it supplies reactive power, negative `q`.

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    AsynchronousMachineBuilder,
    BaseVoltageBuilder,
    BatteryUnitBuilder,
    ElectricalVehicleUnitBuilder,
    EnergySourceBuilder,
    GenerationInverterInfoBuilder,
    PhotoVoltaicUnitBuilder,
    PowerElectronicsConnectionBuilder,
    SolarPanelInfoBuilder,
    SynchronousMachineBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models",
    model="sources_and_generation",
    Feeder="fdr_demo",
)
container = network.container
base_voltage_builder = BaseVoltageBuilder(network, container)
for mRID, nominalVoltage in [
    ("bv_115kV", Qty(115, "kV")),
    ("bv_12_47kV", Qty(12.47, "kV")),
    ("bv_480V", Qty(480, "V")),
]:
    base_voltage_builder.create(mRID=mRID, name=mRID, nominalVoltage=nominalVoltage)

## The grid source

An `EnergySource` is a Thevenin equivalent: a voltage behind an impedance. `add_electrical` sets the
positive-sequence impedance, `add_short_circuit` the zero- and negative-sequence impedances, and `add_ssh`
the source voltage.

In [2]:
source_builder = EnergySourceBuilder(network, container)
source = source_builder.create(mRID="src_grid", name="grid")
source, terminal, source_phases = source_builder.add_connectivity(
    source,
    node="sourcebus",
    nominalVoltage=Qty(115, "kV"),
    phases="ABC",
    BaseVoltage="bv_115kV",
)
source_builder.add_electrical(source, r=Qty(0.5, "ohm"), x=Qty(5, "ohm"))
source_builder.add_short_circuit(
    source,
    r0=Qty(1.5, "ohm"),
    x0=Qty(15, "ohm"),
    r2=Qty(0.5, "ohm"),
    x2=Qty(5, "ohm"),
)
source_builder.add_ssh(
    source, voltageMagnitude=Qty(117.3, "kV"), voltageAngle=Qty(0, "deg")
)

print(
    [phase.mRID for phase in source_phases],
    f"{source.voltageMagnitude.to('kV'):g} kV",
)

['src_grid_A', 'src_grid_B', 'src_grid_C'] 117.3 kV


## Rotating machines

A synchronous generator connects to one node. `add_electrical` sets its rating and its reactive power and
voltage limits, `add_ssh` its operating point and mode, and `add_short_circuit` its fault data. An
asynchronous (induction) machine is built the same way, with its rating, efficiency and kind, `generator`
or `motor`, in `add_electrical`.

In [3]:
generator_builder = SynchronousMachineBuilder(network, container)
generator = generator_builder.create(mRID="gen_1", name="gen 1")
generator, terminal = generator_builder.add_connectivity(
    generator, node="gen_1_bus", BaseVoltage="bv_12_47kV"
)
generator_builder.add_electrical(
    generator, ratedS=Qty(2.5, "MVA"), minQ=Qty(-1, "MVAr"), maxQ=Qty(1.2, "MVAr")
)
generator_builder.add_ssh(
    generator, p=Qty(-2, "MW"), q=Qty(-0.5, "MVAr"), operatingMode="generator"
)
generator_builder.add_short_circuit(
    generator,
    ratedU=Qty(12.47, "kV"),
    ratedPowerFactor=0.9,
    r=0.01,
    x0=0.05,
    x2=0.15,
    earthing=True,
)

induction_builder = AsynchronousMachineBuilder(network, container)
induction = induction_builder.create(mRID="wtg_1", name="wind turbine 1")
induction_builder.add_connectivity(induction, node="wtg_1_bus", BaseVoltage="bv_480V")
induction_builder.add_electrical(
    induction,
    ratedS=Qty(750, "kVA"),
    efficiency=96.5,
    asynchronousMachineType="generator",
)
induction_builder.add_ssh(induction, p=Qty(-600, "kW"))

print(
    generator.mRID,
    generator.operatingMode.value,
    f"rated {generator.ratedS.to('MVA'):g} MVA",
)
print(f"  p = {generator.p.to('MW'):g} MW")
print(f"  q from {generator.minQ.to('MVAr'):g} to {generator.maxQ.to('MVAr'):g} MVAr")
print(
    induction.mRID,
    induction.asynchronousMachineType.value,
    f"rated {induction.ratedS.to('kVA'):g} kVA",
)
print(
    f"  p = {induction.p.to('kW'):g} kW, efficiency {float(induction.efficiency):g} %"
)

gen_1 generator rated 2.5 MVA
  p = -2 MW
  q from -1 to 1.2 MVAr
wtg_1 generator rated 750 kVA
  p = -600 kW, efficiency 96.5 %


## A PV system

An inverter-based resource has two parts:

- the `PowerElectronicsConnection`, the inverter on the grid: its Terminal, rating and output,
- one or more `PowerElectronicsUnit`s behind it, the DC source: here a `PhotoVoltaicUnit`.

The inverter's datasheet is a `GenerationInverterInfo`, and the panel's a `SolarPanelInfo`. Each links to
the equipment it describes with `add_asset_datasheet`.

In [4]:
inverter_builder = PowerElectronicsConnectionBuilder(network, container)
inverter = inverter_builder.create(mRID="pv_1", name="pv 1")
inverter, terminal, inverter_phases = inverter_builder.add_connectivity(
    inverter,
    node="pv_1_bus",
    phaseConnection="Y",
    phases="ABC",
    BaseVoltage="bv_480V",
)
inverter_builder.add_electrical(
    inverter,
    ratedS=Qty(250, "kVA"),
    ratedU=Qty(480, "V"),
    minQ=Qty(-110, "kVAr"),
    maxQ=Qty(110, "kVAr"),
)
inverter_builder.add_ssh(inverter, p=Qty(-200, "kW"), q=Qty(0, "kVAr"))

pv_builder = PhotoVoltaicUnitBuilder(network, container)
array = pv_builder.create(mRID="pv_1_array", name="pv 1 array")
pv_builder.add_electrical(
    array, minP=Qty(0, "kW"), maxP=Qty(250, "kW"), PowerElectronicsConnection="pv_1"
)
pv_builder.add_ssh(array, p=Qty(-200, "kW"), irradiance=Qty(800, "W/m^2"))

inverter_info_builder = GenerationInverterInfoBuilder(network, container)
inverter_info = inverter_info_builder.create(
    mRID="inv_250kVA", name="250 kVA string inverter"
)
inverter_info_builder.add_asset_info(
    inverter_info,
    phaseCount="threePhase",
    ratedVoltage=Qty(480, "V"),
    apparentPowerMaximum=Qty(250, "kVA"),
)
inverter_builder.add_asset_datasheet(inverter, AssetDatasheet="inv_250kVA")

panel_builder = SolarPanelInfoBuilder(network, container)
panel = panel_builder.create(mRID="panel_400W", name="400 W module")
panel_builder.add_asset_info(
    panel,
    efficiency=20.5,
    referenceIrradiation=Qty(1, "kW/m^2"),
    referenceTemperature=Qty(25, "degC"),
)
pv_builder.add_asset_datasheet(array, AssetDatasheet="panel_400W")

print(
    [unit.mRID for unit in inverter.PowerElectronicsUnit],
    inverter.AssetDatasheet.name,
    array.AssetDatasheet.name,
)

['pv_1_array'] 250 kVA string inverter 400 W module


## A battery

A battery is a `BatteryUnit` behind its own inverter, with its energy rating and charge and discharge
limits. Negative `minP` is the charging limit, as charging draws power into the battery's Terminal.
`add_ssh` sets its output, stored energy and state. A `BatteryUnit` has no `q`: its inverter carries the
reactive power.

In [5]:
battery_inverter = inverter_builder.create(mRID="bess_1", name="bess 1")
inverter_builder.add_connectivity(
    battery_inverter,
    node="pv_1_bus",
    phaseConnection="Y",
    phases="ABC",
    BaseVoltage="bv_480V",
)
inverter_builder.add_electrical(
    battery_inverter, ratedS=Qty(125, "kVA"), ratedU=Qty(480, "V")
)

battery_builder = BatteryUnitBuilder(network, container)
battery = battery_builder.create(mRID="bess_1_battery", name="bess 1 battery")
battery_builder.add_electrical(
    battery,
    minP=Qty(-125, "kW"),
    maxP=Qty(125, "kW"),
    ratedE=Qty(500, "kWh"),
    minimumE=Qty(50, "kWh"),
    chargingEfficiency=0.95,
    dischargingEfficiency=0.95,
    PowerElectronicsConnection="bess_1",
)
battery_builder.add_ssh(
    battery, p=Qty(-100, "kW"), storedE=Qty(320, "kWh"), batteryState="discharging"
)

print(
    battery.mRID,
    "behind",
    battery.PowerElectronicsConnection.mRID,
    battery.batteryState.value,
)
print(f"{battery.storedE.to('kWh'):g} of {battery.ratedE.to('kWh'):g} kWh stored")
print(f"p = {battery.p.to('kW'):g} kW")

bess_1_battery behind bess_1 discharging
320 of 500 kWh stored
p = -100 kW


## An electric vehicle

An `ElectricalVehicleUnit` is a battery too, behind its charger, with the same electrical data plus
`v2gCapable`, whether it can discharge into the grid. cim18v17 has no steady-state hypothesis part for it,
so it has no `add_ssh`.

In [6]:
charger = inverter_builder.create(mRID="evse_1", name="charger 1")
inverter_builder.add_connectivity(
    charger, node="pv_1_bus", phaseConnection="Y", phases="ABC", BaseVoltage="bv_480V"
)
inverter_builder.add_electrical(
    charger, ratedS=Qty(19.2, "kVA"), ratedU=Qty(480, "V")
)

ev_builder = ElectricalVehicleUnitBuilder(network, container)
vehicle = ev_builder.create(mRID="evse_1_vehicle", name="vehicle at charger 1")
ev_builder.add_electrical(
    vehicle,
    minP=Qty(-19.2, "kW"),
    maxP=Qty(11.5, "kW"),
    ratedE=Qty(75, "kWh"),
    v2gCapable=True,
    PowerElectronicsConnection="evse_1",
)
print(
    type(vehicle).__name__,
    vehicle.mRID,
    f"{vehicle.ratedE.to('kWh'):g} kWh, v2g {vehicle.v2gCapable}",
)

ElectricalVehicleUnit evse_1_vehicle 75 kWh, v2g True
